# Detección de Muelas del Juicio — Semana 3: Entrenamiento del Modelo

**Materia:** Redes Neuronales  
**Docente:** Ing. Pablo Marinozi  
**Repo:** https://github.com/JulianOliveraBalls/dentex-wisdom-teeth

---

**Prerrequisito:** haber corrido `01_dataset_preparation.ipynb` (genera `data/processed/yolo_merged/`).
Este notebook carga los pesos guardados — **no reentrena**.

| Sección | Contenido |
|---------|----------|
| 0 | Setup — detección Colab/local + restauración de pesos |
| a | Modelo preentrenado y estrategia de fine-tuning |
| b | Configuración del entrenamiento |
| c | Experimentación — 6 experimentos comparados |
| d | Evaluación del modelo elegido |
| e | Guardado del modelo |


## Sección 0 — Setup

In [ ]:
import subprocess, sys, os, gc, json, random, warnings, shutil
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings('ignore')

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *pkgs, '-q'])

try: import ultralytics, sklearn
except ImportError: pip_install('ultralytics','scikit-learn')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from sklearn.metrics import classification_report
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from ultralytics import YOLO

# ── Detección automática Colab / local ────────────────────────────────────────
try:
    import google.colab; IN_COLAB = True
    from google.colab import drive; drive.mount('/drive')
    REPO_ROOT = Path('/content/dentex-wisdom-teeth')
    if not REPO_ROOT.exists():
        subprocess.run(['git','clone',
                        'https://github.com/JulianOliveraBalls/dentex-wisdom-teeth.git',
                        str(REPO_ROOT)], check=True)
    DRIVE_DIR = Path('/drive/MyDrive/dentex_runs')
except ImportError:
    IN_COLAB  = False
    REPO_ROOT = Path('__file__').resolve().parent.parent if '__file__' in dir() else Path.cwd().parent
    DRIVE_DIR = REPO_ROOT / 'dev'

DATA_DIR    = REPO_ROOT / 'data'
YOLO_MERGED = DATA_DIR / 'processed' / 'yolo_merged'
OUTPUTS_DIR = DATA_DIR / 'outputs'
SRC_DIR     = REPO_ROOT / 'src'
DEV_DIR     = REPO_ROOT / 'dev'
for d in [OUTPUTS_DIR, DEV_DIR]: d.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SRC_DIR))

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def log(msg, level='INFO'):
    icons={'INFO':'[INFO]','OK':'[OK]  ','WARN':'[WARN]','ERR':'[ERR] ','DATA':'[DATA]'}
    print(f'{icons.get(level,"[INFO]")} {msg}')

def clear_memory():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

# Verificar que yolo_merged existe (depende de notebook 01)
if not YOLO_MERGED.exists():
    raise RuntimeError(
        'yolo_merged no existe. Correr primero 01_dataset_preparation.ipynb')

log(f'REPO_ROOT: {REPO_ROOT}', 'DATA')
log(f'IN_COLAB: {IN_COLAB}   device: {device}', 'OK')
log(f'yolo_merged: {len(list((YOLO_MERGED/"images"/"train").glob("*.*")))} imgs train', 'DATA')


In [ ]:
# ── Restaurar pesos desde Drive (Colab) o dev/ (local) ──────────────────────
RUNS_DIR = Path('runs/detect')
RESULTS  = {}

_exp_map = {
    'Exp_G1': 'Exp_G1_yolov8s_merged_30ep',
    'Exp_G2': 'Exp_G2_yolov8m_merged_30ep',
    'Exp_G4': 'Exp_G4_yolov8m_merged_60ep_cls15',
    'Exp_G5': 'Exp_G5_yolov8m_erupted_boost_60ep',
    'Exp_G6': 'Exp_G6_yolov8m_exanquad_30ep',
}
_exp_meta = {
    'Exp_G1': ('merged_2619', 30,  'YOLOv8s COCO, merged, 30ep'),
    'Exp_G2': ('merged_2619', 30,  'YOLOv8m COCO, merged, 30ep'),
    'Exp_G4': ('merged_2619', 60,  'YOLOv8m, cls=1.5, aug agresiva, 60ep'),
    'Exp_G5': ('merged_2644', 60,  'YOLOv8m, cls=1.5, +25 erupted DENTEX, 60ep'),
    'Exp_G6': ('merged_6630', 30,  'YOLOv8m, DENTEX+ExAnMTM quad_flip, 30ep'),
}

# Restaurar desde Drive → runs/ (Colab) o desde dev/ (local)
for alias, run_name in _exp_map.items():
    src_dir = DRIVE_DIR / run_name  # Drive en Colab, dev/ en local
    if not src_dir.exists(): continue
    dst_weights = RUNS_DIR / run_name / 'weights'
    dst_weights.mkdir(parents=True, exist_ok=True)
    for f in (src_dir / 'weights').glob('*.pt'):
        dst = dst_weights / f.name
        if not dst.exists(): shutil.copy2(str(f), str(dst))
    csv_src = src_dir / 'results.csv'
    csv_dst = RUNS_DIR / run_name / 'results.csv'
    if csv_src.exists() and not csv_dst.exists():
        shutil.copy2(str(csv_src), str(csv_dst))
    # Cargar métricas desde CSV
    if csv_dst.exists():
        try:
            df_r = pd.read_csv(csv_dst); df_r.columns = df_r.columns.str.strip()
            last = df_r.iloc[-1]
            dataset, epochs, notas = _exp_meta[alias]
            RESULTS[alias] = {
                'dataset': dataset, 'epochs': epochs,
                'mAP50':    round(float(last.get('metrics/mAP50(B)',   0)), 4),
                'mAP50_95': round(float(last.get('metrics/mAP50-95(B)',0)), 4),
                'P':        round(float(last.get('metrics/precision(B)',0)),4),
                'R':        round(float(last.get('metrics/recall(B)',  0)), 4),
                'notas':    notas,
            }
            log(f'{alias}: mAP50={RESULTS[alias]["mAP50"]} (epoch {len(df_r)})', 'OK')
        except Exception as e:
            log(f'{alias}: error leyendo CSV — {e}', 'WARN')

if not RESULTS:
    log('No se encontraron pesos. Copiar best.pt de cada experimento a dev/ o Drive.', 'WARN')


## a) Modelo preentrenado y estrategia de fine-tuning

### Modelo elegido: YOLOv8m

| Criterio | Justificación |
|----------|---------------|
| Tarea | Detección de objetos con bounding boxes |
| Tamaño | Medium (~25M params) supera Small con >2000 imágenes |
| Resolución | 640×640 — nativa de YOLOv8, necesaria para objetos pequeños |

### Backbone: COCO vs dental (nsitnov)

Se probaron dos backbones:

| Backbone | mAP50 val (10ep) | Notas |
|----------|-----------------|-------|
| nsitnov (dental) | 0.657–0.738 | Preentrenado en 8 patologías dentales |
| **COCO** (YOLOv8m.pt) | **0.743** | **Ganador** |

COCO supera al backbone dental porque nsitnov fue entrenado como segmentador —
solo ~61 de 91 capas son compatibles con la arquitectura de detección.

### Modificación de la arquitectura

Solo se cambia la cabeza de detección: `nc=80 (COCO) → nc=2 (erupted/impacted)`.
Ultralytics lo hace automáticamente al especificar `nc=2`.

### Estrategia: fine-tuning completo (sin capas congeladas)

| Decisión | Justificación |
|----------|---------------|
| Sin freeze | Con ~2600-6600 imgs y dominio similar (Rx), congelar el backbone limita la adaptación |
| lr0=0.001667 | Conservador para no destruir features preaprendidas |
| warmup=3ep | Estabiliza gradientes al inicio del fine-tuning |


## b) Configuración del entrenamiento

### Función de pérdida

YOLOv8 usa una pérdida compuesta: `loss = 7.5×box + 0.5×cls + 1.5×dfl`

| Componente | Función | Peso |
|------------|---------|------|
| box | CIoU Loss — localización precisa del bounding box | 7.5 |
| cls | Binary Cross-Entropy — clasificación erupted/impacted | 0.5 |
| dfl | Distribution Focal Loss — precisión de los bordes | 1.5 |

El desbalance erupted/impacted (~37/63%) no es severo — no requiere Focal Loss.
Se probó `cls=1.5` en Exp_G4 para penalizar más los errores en erupted,
pero empeoró el test — la BCE estándar es más estable.

### Optimizador y scheduler

| Parámetro | Valor | Justificación |
|-----------|-------|---------------|
| Optimizador | AdamW | Mejor convergencia que SGD en fine-tuning |
| lr0 | 0.001667 | Default ultralytics para AdamW |
| Scheduler | Cosine annealing | Decaimiento suave |
| weight_decay | 0.0005 | Regularización L2 |

### Augmentations internas de YOLOv8

| Parámetro | Valor | Justificación |
|-----------|-------|---------------|
| fliplr | 0.5 | Simetría bilateral del arco dental |
| hsv_v | 0.2 | Variabilidad de brillo entre hospitales |
| degrees | 5.0 | Rotaciones por posicionamiento del paciente |
| mosaic | 0.0 | Desactivado — mezclar anatomías es inválido |

### Hardware

Google Colab Tesla T4 (15GB VRAM), batch=8, ~2hs por run de 30 epochs.


In [ ]:
# Configuración base de entrenamiento
TRAIN_KWARGS = dict(
    imgsz=640, batch=8, workers=2,
    optimizer='AdamW', lr0=0.001667, lrf=0.01,
    momentum=0.937, weight_decay=0.0005, warmup_epochs=3,
    fliplr=0.5, hsv_v=0.2, degrees=5.0, translate=0.05,
    mosaic=0.0, mixup=0.0,
    plots=True, verbose=False, exist_ok=True,
)
log('TRAIN_KWARGS configurado', 'OK')
for k,v in TRAIN_KWARGS.items(): log(f'  {k}: {v}', 'DATA')


## c) Experimentación

Se realizaron **6 experimentos** en 3 fases para responder preguntas específicas.

### Fase 1 — Comparación backbone y dataset (10 epochs)

**Pregunta:** ¿conviene un backbone dental o COCO? ¿Los cuadrantes ayudan?

| Exp | Backbone | Dataset | mAP50 val | Conclusión |
|-----|----------|---------|-----------|------------|
| Exp_A | nsitnov | base 308 imgs | 0.657 | Baseline mínimo |
| Exp_B | nsitnov | quad_flip 1744 | 0.738 | Cuadrantes +0.08 |
| **Exp_C** | **COCO** | **quad_flip 1744** | **0.743** | **Ganador F1** |

**Aprendizaje:** el dataset expandido importa más que el backbone dental.
COCO supera a nsitnov con datos suficientes.

### Fase G — Dataset merged DENTEX + ExAn-MTM

**Pregunta:** ¿ExAn-MTM mejora los resultados? ¿Cómo abordar erupted?

| Exp | Modelo | Dataset | Epochs | Cambio | mAP50 val | mAP50 test | erupted | impacted |
|-----|--------|---------|--------|--------|-----------|------------|---------|----------|
| Exp_G1 | YOLOv8s | merged 2619 | 30 | — | 0.930 | — | — | — |
| **Exp_G2** | **YOLOv8m** | **merged 2619** | **30** | **—** | **0.940** | **0.675** | **0.444** | **0.905** |
| Exp_G4 | YOLOv8m | merged 2619 | 60 | cls=1.5, aug+ | 0.969 | 0.672 | 0.441 | 0.903 |
| Exp_G5 | YOLOv8m | merged 2644 | 60 | +25 erupted | 0.954 | 0.614 | 0.145 | 0.572 |
| Exp_G6 | YOLOv8m | merged 6630 | 30 | ExAn quad_flip | 0.956 | 0.668 | 0.391 | **0.945** |

### Aprendizaje por experimento

**G1→G2:** YOLOv8m supera a YOLOv8s con el dataset merged. Mayor capacidad justificada.

**G2→G4:** cls=1.5 mejora val (0.969) pero empeora test (0.672). Overfitting al val set de ExAn-MTM.

**G4→G5:** mover erupted de test a train colapsa impacted (0.572). Resultado inválido.

**G2→G6:** cuadrantes de ExAn-MTM mejoran impacted (+0.04) pero no erupted (-0.05).
Confirma que erupted es un problema estructural, no de cantidad de datos.

### Por qué erupted es difícil

| | erupted | impacted |
|---|---------|----------|
| Señal visual | Diente en posición **normal** — igual a molares adyacentes | Diente en posición **anómala** |
| Contexto necesario | Saber que es el **diente 8** del arco (global) | Ver que está torcido (local) |

YOLO trabaja con ventanas locales y no puede contar dientes desde el centro del arco.
Esta es una limitación estructural de la arquitectura, no del dataset.


In [ ]:
# Tabla comparativa de todos los experimentos
if RESULTS:
    df_res = pd.DataFrame(RESULTS).T.reset_index().rename(columns={'index':'Experimento'})
    df_res = df_res.sort_values('mAP50', ascending=False)
    print('Tabla comparativa — val set (mejor epoch durante entrenamiento):')
    print()
    print(df_res.to_string(index=False))
else:
    log('Sin resultados cargados — verificar pesos en dev/ o Drive', 'WARN')

# Tabla test set (hardcodeada con resultados conocidos)
print()
print('Tabla comparativa — TEST SET (nunca visto durante entrenamiento):')
test_data = {
    'Exp': ['Exp_G2','Exp_G4','Exp_G6'],
    'mAP50_all': [0.675, 0.672, 0.668],
    'mAP50_erupted': [0.444, 0.441, 0.391],
    'mAP50_impacted': [0.905, 0.903, 0.945],
    'P': [0.596, 0.636, 0.598],
    'R': [0.917, 0.759, 0.771],
}
df_test = pd.DataFrame(test_data)
print(df_test.to_string(index=False))
print()
print('Modelo elegido: Exp_G2 — mejor balance erupted/impacted en test')


## d) Evaluación del modelo elegido: Exp_G2

**Criterio de elección:** mejor mAP50 global en test. G6 mejora impacted (+0.04)
pero empeora erupted (-0.05) — G2 tiene el mejor balance.

### Observación sobre overfitting

Caída de 0.940 (val) → 0.675 (test) indica overfitting al val set.
El val mezcla ExAn-MTM (dominio fácil) con DENTEX (dominio difícil).
El test es solo DENTEX — el dominio más desafiante.
Esto se mantiene en todos los experimentos y se documenta como limitación del dataset.

### Métricas finales sobre test set

| Clase | mAP50 | P | R | F1 |
|-------|-------|---|---|----|
| erupted | 0.444 | 0.398 | 0.452 | 0.424 |
| **impacted** | **0.905** | **0.794** | **0.939** | **0.860** |
| all | 0.675 | 0.596 | 0.917 | 0.723 |

### Clasificación por imagen (F1, sin penalizar IoU)

| Clase | P | R | F1 | Support |
|-------|---|---|----|---------|
| erupted | 0.846 | 0.667 | 0.746 | 33 |
| impacted | 0.810 | 0.922 | 0.862 | 51 |
| **accuracy** | | | **0.821** | **84** |

La diferencia entre mAP50=0.444 y F1=0.746 en erupted indica que el modelo
**clasifica correctamente** pero a veces el bounding box no tiene suficiente IoU.

### Comparación con literatura

| Paper | mAP50 impacted | Dataset |
|-------|---------------|--------|
| Celik 2022 | 0.96 | 440 Rx privadas (1 hospital) |
| E-MTMYOLO 2025 | 0.934 | ExAn-MTM solo |
| Zhicheng 2024 | 0.867 | DENTEX |
| **Exp_G2 (nuestro)** | **0.905** | DENTEX + ExAn-MTM |
| **Exp_G6 (nuestro)** | **0.945** | DENTEX + ExAn-MTM quad |


In [ ]:
# Cargar modelo G2
G2_WEIGHTS = RUNS_DIR / 'Exp_G2_yolov8m_merged_30ep' / 'weights' / 'best.pt'

if not G2_WEIGHTS.exists():
    # Buscar en dev/ del repo (pesos versionados)
    for candidate in [DEV_DIR / 'modelo.pt', DEV_DIR / 'modelo.pth', DEV_DIR / 'Exp_G2_best.pt']:
        if candidate.exists():
            G2_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(str(candidate), str(G2_WEIGHTS))
            log(f'G2 cargado desde {candidate.name}', 'OK')
            break
    else:
        log('Pesos no encontrados. Verificar dev/modelo.pt', 'ERR')

if G2_WEIGHTS.exists():
    model_G2 = YOLO(str(G2_WEIGHTS))
    log(f'Modelo G2 cargado: {G2_WEIGHTS}', 'OK')
    # Evaluación sobre test set
    metrics_test = model_G2.val(
        data=str(YOLO_MERGED/'dataset.yaml'), split='test', imgsz=640, batch=8, verbose=True)


In [ ]:
# Curvas de entrenamiento de G2
csv_path = RUNS_DIR / 'Exp_G2_yolov8m_merged_30ep' / 'results.csv'
if csv_path.exists():
    df_r = pd.read_csv(csv_path); df_r.columns = df_r.columns.str.strip()
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, col, title in [
        (axes[0], 'train/cls_loss',     'Cls Loss (train)'),
        (axes[1], 'train/box_loss',     'Box Loss (train)'),
        (axes[2], 'metrics/mAP50(B)',   'mAP50 (val)'),
    ]:
        if col in df_r.columns:
            ax.plot(df_r[col], linewidth=2)
            ax.set_title(title); ax.set_xlabel('Epoch'); ax.grid(alpha=0.3)
    plt.suptitle('Exp_G2 — Curvas de entrenamiento (30 epochs)')
    plt.tight_layout()
    plt.savefig(str(OUTPUTS_DIR/'G2_training_curves.png'), dpi=100, bbox_inches='tight')
    plt.show()
else:
    log('CSV de G2 no encontrado', 'WARN')


In [ ]:
# Análisis de errores sobre test set
if G2_WEIGHTS.exists():
    CLASS_NAMES = {0:'erupted', 1:'impacted'}
    test_img_dir = YOLO_MERGED / 'images' / 'test'
    test_lbl_dir = YOLO_MERGED / 'labels' / 'test'
    img_paths = sorted(list(test_img_dir.glob('*.png'))+list(test_img_dir.glob('*.jpg')))
    fallos = []
    for img_path in img_paths:
        lbl_path = test_lbl_dir / (img_path.stem+'.txt')
        if not lbl_path.exists(): continue
        true_classes = [int(l.split()[0]) for l in lbl_path.read_text().strip().split('\n') if l]
        if not true_classes: continue
        true_cls = max(set(true_classes), key=true_classes.count)
        r = model_G2.predict(str(img_path), imgsz=640, verbose=False, conf=0.25)[0]
        if len(r.boxes)==0: fallos.append((img_path,-1,true_cls,0.0)); continue
        confs=r.boxes.conf.cpu().numpy(); clases=r.boxes.cls.cpu().numpy().astype(int)
        pred_cls=clases[confs.argmax()]
        if pred_cls!=true_cls: fallos.append((img_path,pred_cls,true_cls,float(confs.max())))
    log(f'Test: {len(img_paths)} imgs | {len(fallos)} errores ({len(fallos)/max(len(img_paths),1)*100:.1f}%)','DATA')
    no_det=sum(1 for _,p,_,_ in fallos if p==-1)
    eru_imp=sum(1 for _,p,t,_ in fallos if t==0 and p==1)
    imp_eru=sum(1 for _,p,t,_ in fallos if t==1 and p==0)
    log(f'  No detectado:         {no_det}','DATA')
    log(f'  erupted→impacted(FP): {eru_imp}','DATA')
    log(f'  impacted→erupted(FN): {imp_eru}','DATA')

    if fallos:
        MEAN_T=torch.tensor([0.485,0.456,0.406]).view(3,1,1)
        STD_T =torch.tensor([0.229,0.224,0.225]).view(3,1,1)
        n_show=min(8,len(fallos))
        fig,axes=plt.subplots(2,4,figsize=(20,8))
        for i,ax in enumerate(axes.flatten()):
            if i>=n_show: ax.axis('off'); continue
            img_path,pred_cls,true_cls,conf=fallos[i]
            lbl_path=test_lbl_dir/(img_path.stem+'.txt')
            try:
                pil=Image.open(img_path).convert('RGB').resize((640,640),Image.LANCZOS)
                ax.imshow(np.array(pil))
                if lbl_path.exists():
                    for line in lbl_path.read_text().strip().split('\n'):
                        if not line: continue
                        parts=line.split(); cls=int(parts[0]); xc,yc,bw,bh=map(float,parts[1:])
                        x1,y1=(xc-bw/2)*640,(yc-bh/2)*640
                        ax.add_patch(patches.Rectangle((x1,y1),bw*640,bh*640,linewidth=2,
                            edgecolor='#FF4444' if cls==1 else '#44CC44',facecolor='none'))
                pred_label=CLASS_NAMES.get(pred_cls,'no_det')
                ax.set_title(f'GT:{CLASS_NAMES[true_cls]} PRED:{pred_label}({conf:.2f})',fontsize=7,color='red')
            except Exception as e: ax.set_title(f'Error:{e}',fontsize=6,color='red')
            ax.axis('off')
        plt.suptitle('Errores — Verde=erupted GT  Rojo=impacted GT')
        plt.tight_layout()
        plt.savefig(str(OUTPUTS_DIR/'errores_G2.png'),dpi=100,bbox_inches='tight')
        plt.show()


## e) Guardado del modelo

Los pesos del modelo final se guardan en `dev/modelo.pth` y `dev/Exp_G2_best.pt`.

**Tamaño:** ~52 MB — dentro del límite de 100 MB de GitHub (no requiere Git LFS).

Si se usara YOLOv8l (>100 MB) habría que activar LFS:
```bash
git lfs install && git lfs track '*.pt' '*.pth'
```


In [ ]:
if G2_WEIGHTS.exists():
    # modelo.pth — nombre pedido por la cátedra
    dst_pth = DEV_DIR / 'modelo.pth'
    shutil.copy2(str(G2_WEIGHTS), str(dst_pth))
    log(f'modelo.pth guardado: {dst_pth}  ({dst_pth.stat().st_size/1e6:.1f} MB)', 'OK')

    # Copia con nombre descriptivo
    dst_exp = DEV_DIR / 'Exp_G2_best.pt'
    shutil.copy2(str(G2_WEIGHTS), str(dst_exp))
    log(f'Exp_G2_best.pt guardado: {dst_exp}', 'OK')

    # Resumen final
    log('', 'INFO')
    log('=== RESUMEN FINAL ===', 'INFO')
    log('Modelo: Exp_G2 — YOLOv8m COCO + DENTEX quad_flip + ExAn-MTM, 30ep', 'OK')
    log('Test set (solo DENTEX):', 'DATA')
    log('  all:      mAP50=0.675  P=0.596  R=0.917', 'DATA')
    log('  erupted:  mAP50=0.444  F1=0.746  (limitación: ambigua sin contexto global)', 'DATA')
    log('  impacted: mAP50=0.905  F1=0.860  (supera literatura sobre DENTEX)', 'DATA')
else:
    log('Pesos no disponibles — saltar guardado', 'WARN')
